# HypotheSAEs Quickstart: PubMed Cancer Prediction

This notebook demonstrates HypotheSAEs on the PubMed 20k RCT dataset to answer: **"What features of a patient's clinical notes predict if they will develop cancer?"**

We'll use medical abstracts from PubMed to identify features that predict whether a study is cancer-related.
- We use a sentence-transformers model for text embeddings. This can run on CPU or GPU (if a cuda device is available, it will use GPU).
- We use a large language model loaded with transformers for hypothesis generation and text annotation. This requires GPU.
- This notebook uses `Qwen/Qwen2.5-3B-Instruct`. (If your GPU doesn't support this model or have enough memory, you can use a different model.)
- Takes about 30-40 minutes to run using an NVIDIA A6000.

**Dataset**: PubMed 20k RCT contains medical abstracts with sections: BACKGROUND, OBJECTIVE, METHODS, RESULTS, CONCLUSIONS.
**Task**: Binary classification to predict if an abstract is cancer-related based on its content.


In [ ]:
%load_ext autoreload
%autoreload 2

import os
# os.environ['CUDA_VISIBLE_DEVICES'] = '6' 

import os, multiprocessing as mp

os.environ["CUDA_VISIBLE_DEVICES"] = "5"

os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

os.environ["VLLM_USE_FLASHINFER"] = "0"

mp.set_start_method("spawn", force=True)
import numpy as np
import pandas as pd
import re
from collections import Counter

from hypothesaes.quickstart import train_sae, interpret_sae, generate_hypotheses, evaluate_hypotheses
from hypothesaes.embedding import get_local_embeddings
from hypothesaes.llm_local import get_vllm_engine

**0. Load and prepare PubMed b574**

We'll load the PubMed 20k RCT dataset and create a cancer prediction task by:
1. Combining abstract sections into full abstracts
2. Creating binary labels based on cancer-related keywords
3. Splitting into train/validation/holdout sets


In [ ]:
# Download PubMed dataset if not already available
import kagglehub

# Download latest version
dataset_path = kagglehub.dataset_download("matthewjansen/pubmed-200k-rtc")
print("Path to dataset files:", dataset_path)

# Load the dataset
train_df = pd.read_csv(os.path.join(dataset_path, "PubMed_20k_RCT/train.csv"))
dev_df = pd.read_csv(os.path.join(dataset_path, "PubMed_20k_RCT/dev.csv"))
test_df = pd.read_csv(os.path.join(dataset_path, "PubMed_20k_RCT/test.csv"))

print(f"Train: {len(train_df)} rows")
print(f"Dev: {len(dev_df)} rows")
print(f"Test: {len(test_df)} rows")
print(f"Target labels: {train_df['target'].unique()}")


Path to dataset files: /b574/qingpengkong/.cache/kagglehub/datasets/matthewjansen/pubmed-200k-rtc/versions/5
Train: 180040 rows
Dev: 30212 rows
Test: 30135 rows
Target labels: ['OBJECTIVE' 'METHODS' 'RESULTS' 'CONCLUSIONS' 'BACKGROUND']


In [ ]:
# Combine abstract sections into full abstracts
def combine_abstract_sections(df):
    """Combine sections of the same abstract into full abstracts"""
    abstracts = {}
    
    for _, row in df.iterrows():
        abstract_id = row['abstract_id']
        text = row['abstract_text']
        section = row['target']
        
        if abstract_id not in abstracts:
            abstracts[abstract_id] = {
                'text': '',
                'sections': []
            }
        
        abstracts[abstract_id]['sections'].append((section, text))
    
    # Combine sections in order
    combined_abstracts = []
    for abstract_id, data in abstracts.items():
        # Sort sections by their typical order
        section_order = {'BACKGROUND': 0, 'OBJECTIVE': 1, 'METHODS': 2, 'RESULTS': 3, 'CONCLUSIONS': 4}
        sorted_sections = sorted(data['sections'], key=lambda x: section_order.get(x[0], 5))
        
        # Combine text
        full_text = ' '.join([text for _, text in sorted_sections])
        combined_abstracts.append({
            'abstract_id': abstract_id,
            'text': full_text,
            'sections': [s[0] for s in sorted_sections]
        })
    
    return pd.DataFrame(combined_abstracts)

# Combine sections for each dataset
train_abstracts = combine_abstract_sections(train_df)
dev_abstracts = combine_abstract_sections(dev_df)
test_abstracts = combine_abstract_sections(test_df)

print(f"Combined abstracts - Train: {len(train_abstracts)}, Dev: {len(dev_abstracts)}, Test: {len(test_abstracts)}")
print(f"Sample abstract length: {len(train_abstracts.iloc[0]['text'])} characters")


Combined abstracts - Train: 15000, Dev: 2500, Test: 2500
Sample abstract length: 2019 characters


In [ ]:
# Create cancer prediction labels
def is_cancer_related(text):
    """Determine if abstract is cancer-related based on keywords"""
    cancer_keywords = [
        'cancer', 'carcinoma', 'tumor', 'tumour', 'neoplasm', 'malignancy', 'malignant',
        'oncology', 'oncological', 'metastasis', 'metastatic', 'chemotherapy', 'radiation',
        'sarcoma', 'lymphoma', 'leukemia', 'leukaemia', 'melanoma', 'adenocarcinoma',
        'breast cancer', 'lung cancer', 'prostate cancer', 'colorectal cancer', 'pancreatic cancer',
        'ovarian cancer', 'cervical cancer', 'liver cancer', 'brain cancer', 'skin cancer',
        'cancer cell', 'cancerous', 'tumorigenesis', 'carcinogenesis', 'biomarker', 'biopsy'
    ]
    
    text_lower = text.lower()
    
    # Check for cancer keywords
    for keyword in cancer_keywords:
        if keyword in text_lower:
            return True
    
    return False

# Create labels
train_abstracts['is_cancer'] = train_abstracts['text'].apply(is_cancer_related)
dev_abstracts['is_cancer'] = dev_abstracts['text'].apply(is_cancer_related)
test_abstracts['is_cancer'] = test_abstracts['text'].apply(is_cancer_related)

# Print statistics
print("Cancer vs Non-cancer distribution:")
print(f"Train: {train_abstracts['is_cancer'].sum()}/{len(train_abstracts)} cancer-related ({train_abstracts['is_cancer'].mean():.2%})")
print(f"Dev: {dev_abstracts['is_cancer'].sum()}/{len(dev_abstracts)} cancer-related ({dev_abstracts['is_cancer'].mean():.2%})")
print(f"Test: {test_abstracts['is_cancer'].sum()}/{len(test_abstracts)} cancer-related ({test_abstracts['is_cancer'].mean():.2%})")

# Sample some cancer-related abstracts
print("\nSample cancer-related abstracts:")
cancer_samples = train_abstracts[train_abstracts['is_cancer']].head(3)
for i, (_, row) in enumerate(cancer_samples.iterrows()):
    print(f"\n{i+1}. Abstract ID: {row['abstract_id']}")
    print(f"Text preview: {row['text'][:200]}...")


Cancer vs Non-cancer distribution:
Train: 2287/15000 cancer-related (15.25%)
Dev: 401/2500 cancer-related (16.04%)
Test: 421/2500 cancer-related (16.84%)

Sample cancer-related abstracts:

1. Abstract ID: 24293578
Text preview: To investigate the efficacy of 6 weeks of daily low-dose oral prednisolone in improving pain , mobility , and systemic low-grade inflammation in the short term and whether the effect would be sustaine...

2. Abstract ID: 24807407
Text preview: The classification of clinical severity of Ebstein anomaly still remains a challenge . The aim of this study was to focus on the interaction of the pathologically altered right heart with the anatomic...

3. Abstract ID: 25231496
Text preview: While overall survival for most common cancers in Australia is improving , the rural-urban differential has been widening , with significant excess deaths due to lung , colorectal , breast and prostat...


In [ ]:
# Prepare data for HypotheSAEs
# Use a subset for faster processing (you can increase these numbers)
MAX_TRAIN = 10000  # Use 10k abstracts for training
MAX_VAL = 2000    # Use 2k abstracts for validation
MAX_HOLDOUT = 2000  # Use 2k abstracts for holdout

# Sample b574
train_sample = train_abstracts.sample(n=min(MAX_TRAIN, len(train_abstracts)), random_state=42)
val_sample = dev_abstracts.sample(n=min(MAX_VAL, len(dev_abstracts)), random_state=42)
holdout_sample = test_abstracts.sample(n=min(MAX_HOLDOUT, len(test_abstracts)), random_state=42)

# Extract texts and labels
texts = train_sample['text'].tolist()
labels = train_sample['is_cancer'].astype(int).values
val_texts = val_sample['text'].tolist()
holdout_texts = holdout_sample['text'].tolist()
holdout_labels = holdout_sample['is_cancer'].astype(int).values

print(f"Final dataset sizes:")
print(f"Train: {len(texts)} abstracts, {labels.sum()} cancer-related ({labels.mean():.2%})")
print(f"Val: {len(val_texts)} abstracts")
print(f"Holdout: {len(holdout_texts)} abstracts, {holdout_labels.sum()} cancer-related ({holdout_labels.mean():.2%})")


Final dataset sizes:
Train: 10000 abstracts, 1515 cancer-related (15.15%)
Val: 2000 abstracts
Holdout: 2000 abstracts, 334 cancer-related (16.70%)


**1. Compute text embeddings for your dataset**

We'll compute text embeddings for medical abstracts using a sentence-transformers model.
We'll use `nomic-ai/modernbert-embed-base` which works well for medical text.


In [ ]:
EMBEDDER = "nomic-ai/modernbert-embed-base"
CACHE_NAME = f"pubmed_cancer_{EMBEDDER}"

text2embedding = get_local_embeddings(texts + val_texts, model=EMBEDDER, batch_size=128, cache_name=CACHE_NAME)
embeddings = np.stack([text2embedding[text] for text in texts])

train_embeddings = np.stack([text2embedding[text] for text in texts])
val_embeddings = np.stack([text2embedding[text] for text in val_texts])

print(f"Embeddings shape: {embeddings.shape}")
print(f"Train embeddings shape: {train_embeddings.shape}")
print(f"Val embeddings shape: {val_embeddings.shape}")


Loading embedding chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Loaded 12000 embeddings in 0.1s
Embeddings shape: (10000, 768)
Train embeddings shape: (10000, 768)
Val embeddings shape: (2000, 768)


**2. Train SAE**

We will train a Matryoshka SAE with $M=256$, $k=8$, and $\\text{prefix\\_lengths} = [32, 256]$.

With the Matryoshka loss, the SAE will learn to reconstruct the input from (1) just the first 32 neurons, and (2) all 256 neurons.
This will produce 32 coarse-grained features, and 224 finer-grained features.


In [ ]:
current_dir = os.getcwd()
if current_dir.endswith("notebooks"):
    prefix = "../"
else:
    prefix = "./"

checkpoint_dir = os.path.join(prefix, "checkpoints", CACHE_NAME)
sae = train_sae(embeddings=train_embeddings, M=256, K=8, matryoshka_prefix_lengths=[32, 256], checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings)


Loaded model from ../checkpoints/pubmed_cancer_nomic-ai/modernbert-embed-base/SAE_matryoshka_M=256_K=8_prefixes=32-256.pt onto device cuda


**3. Interpret neurons**

This step will load a local LLM with transformers, and use it to interpret a few random neurons in the SAE.
This is to ensure that the local LLM is working and the interpretations look reasonable for medical text.


In [ ]:
'''
Some vLLM notes:
- When running in notebook, it's best to use the same interpreter and annotator model;
  this is because there is some memory overhead when switching models (ie we can't fully free GPU memory).
- We set `gpu_memory_utilization=0.85`, which avoids OOMs with Qwen3-32B-AWQ on an A6000.
  You may want to adjust this depending on your hardware/model.
- You can increase `tensor_parallel_size` to use multiple GPUs on the same node.
  See vLLM docs: https://docs.vllm.ai/en/latest/serving/distributed_serving.html
'''

INTERPRETER_MODEL = ANNOTATOR_MODEL = "Qwen/Qwen3-8B-AWQ"

engine = get_vllm_engine(
    INTERPRETER_MODEL, 
    gpu_memory_utilization=0.7,
    quantization="awq", 
    tensor_parallel_size=1,
)

Loading Qwen/Qwen3-8B-AWQ in vLLM...
WARNING 09-07 10:28:14 [__init__.py:520] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


`torch_dtype` is deprecated! Use `dtype` instead!


WARNING 09-07 10:28:23 [__init__.py:1171] awq quantization is not fully optimized yet. The speed can be slower than non-quantized models.
WARNING 09-07 10:28:29 [topk_topp_sampler.py:61] FlashInfer is not available. Falling back to the PyTorch-native implementation of top-p & top-k sampling. For the best performance, please install FlashInfer.


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:09<00:00,  7.15it/s]


Loaded Qwen/Qwen3-8B-AWQ with dtype: torch.float16 (took 51.6s)


In [ ]:

TASK_SPECIFIC_INSTRUCTIONS = """All of the texts are medical abstracts from PubMed.
Features should describe specific aspects of medical research that might be relevant to cancer prediction. For example:
- "mentions specific cancer types or oncology terminology"
- "describes treatment protocols involving chemotherapy or radiation"
- "discusses biomarkers or biagnostic methods for cancer detection"
- "references clinical trials involving cancer patients"
- "mentions tumor characteristics or cancer staging"

Focus on medical and scientific terminology that would be characteristic of cancer-related research."""


**4. Generate hypotheses**

Generate hypotheses which are predictive of whether an abstract is cancer-related.

We use the local LLM to generate interpretations and then estimate each interpretation's fidelity by annotating texts with the interpretation.

Here, we select neurons after ranking them by correlation with the target variable (cancer vs non-cancer).


In [ ]:
# Generate hypotheses (only if vLLM engine is available)
if engine is not None:
    selection_method = "correlation"
    results = generate_hypotheses(
        texts=texts,
        labels=labels,
        embeddings=embeddings,
        sae=sae,
        interpreter_model=INTERPRETER_MODEL,
        annotator_model=ANNOTATOR_MODEL,
        selection_method=selection_method,
        n_selected_neurons=20,
        n_candidate_interpretations=1,
        task_specific_instructions=TASK_SPECIFIC_INSTRUCTIONS
    )

    print("\nMost predictive features of cancer-related medical abstracts:")
    pd.set_option('display.max_colwidth', None)
    display(results.sort_values(by=f"target_{selection_method}", ascending=False))
    pd.reset_option('display.max_colwidth')
    print("\n✅ Hypothesis generation completed!")
else:
    print("❌ Cannot generate hypotheses - vLLM engine not available")


Embeddings shape: (10000, 768)


Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Activations shape: (10000, 256)

Step 1: Selecting top 20 predictive neurons

Step 2: Interpreting selected neurons


Adding requests:   0%|          | 0/20 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/20 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Adding requests:   0%|          | 0/2000 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/2000 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…


Most predictive features of cancer-related medical abstracts:


,neuron_idx,source_sae,target_correlation,interpretation,f1_fidelity_score
0,1,"(SAE_0, 256, 8)",0.761779,mentions specific cancer types or oncology terminology,0.591549
1,137,"(SAE_0, 256, 8)",0.281540,mentions overall survival (OS) and progression-free survival (PFS) as primary endpoints in clinical trials for cancer treatment,0.823529
2,140,"(SAE_0, 256, 8)",0.265089,mentions specific cancer types or oncology terminology,0.564706
3,80,"(SAE_0, 256, 8)",0.263461,describes treatment protocols involving chemotherapy or radiation,0.958333
4,129,"(SAE_0, 256, 8)",0.228278,mentions non-small cell lung cancer (NSCLC) as the primary condition,0.927312
5,112,"(SAE_0, 256, 8)",0.218604,mentions EGFR mutations or related therapeutic targets,0.901099
6,236,"(SAE_0, 256, 8)",0.189980,mentions specific cancer-related survival outcomes such as progression-free survival or overall survival,0.696216
7,253,"(SAE_0, 256, 8)",0.161055,mentions prostate-specific antigen (PSA) levels or Gleason scoring in the context of prostate cancer biagnosis,0.765432
8,101,"(SAE_0, 256, 8)",0.139706,mentions cervical or colorectal cancer screening methods,0.750000
9,34,"(SAE_0, 256, 8)",0.137537,mentions specific cancer-related biomarkers and their association with tumor prognosis or biagnostic methods,0.744304



✅ Hypothesis generation completed!


**5. Evaluate held-out generalization**

Finally, we evaluate whether these are good hypotheses by testing whether their natural language interpretations can predict whether an abstract is cancer-related.

We compute annotations for each hypothesized concept on a holdout set (not seen during SAE training & feature selection).
This step uses the same local LLM as the one used to generate interpretations.


In [ ]:
# Evaluate hypotheses (only if vLLM engine is available)
if engine is not None:
    metrics, evaluation_df = evaluate_hypotheses(
        hypotheses_df=results,
        texts=holdout_texts,
        labels=holdout_labels,
        annotator_model=ANNOTATOR_MODEL,
    )

    pd.set_option('display.max_colwidth', None)
    display(evaluation_df)
    pd.reset_option('display.max_colwidth')

    print("\nHoldout Set Metrics:")
    print(f"R² Score: {metrics['r2']:.3f}")
    print(f"Significant hypotheses: {metrics['Significant'][0]}/{metrics['Significant'][1]} " 
          f"(p < {metrics['Significant'][2]:.3e})")

    # Additional analysis for cancer prediction
    print("\nCancer Prediction Analysis:")
    print(f"Baseline accuracy (always predict majority class): {max(holdout_labels.mean(), 1-holdout_labels.mean()):.3f}")
    print(f"Cancer prevalence in holdout set: {holdout_labels.mean():.2%}")
    print("\n✅ Evaluation completed!")
else:
    print("❌ Cannot evaluate hypotheses - vLLM engine not available")


Step 1: Annotating texts with 20 hypotheses
Found 0 cached items; annotating 40000 uncached items


Adding requests:   0%|          | 0/40000 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/40000 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/…

Step 2: Computing predictiveness of hypothesis annotations
         Current function value: 0.290781
         Iterations: 35
Error fitting model: Singular matrix, trying OLS instead


/b574/qingpengkong/miniconda3/envs/hypo/lib/python3.11/site-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


,hypothesis,separation_score,separation_pval,regression_coef,regression_pval,feature_prevalence
10,mentions low-tube-voltage CT protocols and their impact on signal-to-noise ratio and radiation dose,0.834251,1.061423e-04,0.233909,4.401860e-01,0.0015
17,mentions chemotherapy-induced nausea and vomiting (CINV) as a target condition for antiemetic treatment protocols,0.834251,1.061423e-04,0.449947,9.822101e-03,0.0015
13,mentions low tube voltage protocols combined with iterative reconstruction techniques to reduce radiation dose in CT angiography,0.833834,1.566648e-03,0.430127,2.429831e-01,0.0010
1,mentions overall survival (OS) and progression-free survival (PFS) as primary endpoints in clinical trials for cancer treatment,0.779830,3.596032e-65,0.137642,1.739830e-01,0.0320
12,mentions progression-free survival (PFS) and overall survival (OS) as key endpoints in clinical trials for cancer treatment,0.763255,3.431263e-70,0.184168,6.590281e-02,0.0360
3,mentions non-small cell lung cancer (NSCLC) as the primary condition,0.662107,1.314147e-24,0.329447,2.504279e-09,0.0165
4,mentions EGFR mutations or related therapeutic targets,0.650706,2.774076e-12,-0.086758,2.852250e-01,0.0080
6,mentions prostate-specific antigen (PSA) levels or Gleason scoring in the context of prostate cancer biagnosis,0.587702,2.883970e-10,0.301471,6.690220e-05,0.0080
8,mentions specific cancer-related biomarkers and their association with tumor prognosis or biagnostic methods,0.514236,7.349547e-70,0.341349,6.451261e-39,0.0830
2,describes treatment protocols involving chemotherapy or radiation,0.450875,1.166313e-92,0.232708,5.559129e-27,0.1520



Holdout Set Metrics:
R² Score: 0.373
Significant hypotheses: 6/18 (p < 5.556e-03)

Cancer Prediction Analysis:
Baseline accuracy (always predict majority class): 0.833
Cancer prevalence in holdout set: 16.70%

✅ Evaluation completed!


**Summary**

This notebook demonstrates HypotheSAEs on medical abstracts to identify features that predict cancer-related research.

**Key findings:**
- The SAE learned to identify specific medical terminology and concepts
- The most predictive features likely relate to cancer-specific vocabulary, treatment protocols, and biagnostic methods
- The approach successfully generalizes to unseen medical abstracts

**Potential applications:**
- Automated classification of medical literature
- Identification of cancer-related research patterns
- Discovery of novel biomarkers or treatment approaches
- Medical literature mining and organization

**Next steps:**
- Experiment with different SAE architectures
- Try different medical embedding models
- Apply to other medical classification tasks
- Integrate with clinical decision support systems
